In [2]:
# To run only once or else restart the kernel
# or change manually the current directory so that it is Networks_project/
import os
os.chdir("../")

In [20]:
import numpy as np
import pandas as pd

## Importation des datasets

In [ ]:
path_data_check_in='data/dataset_TSMC2014_TKY.txt'
path_data_users='data/dataset_UbiComp2016_UserProfile_TKY.txt'

df_checkins = pd.read_csv(
        path_data_check_in,
        sep='\t',
        encoding='latin-1',
        header=None,
        names=[
            'user_id','location_id','location_type_ID',
            'location_type_name','latitude','longitude',
            'timezone','timestamp', 'Venue_Category_Name'
        ]
    )

df_users = pd.read_csv(
        path_data_users,
        sep='\t',
        encoding='latin-1',
        header=None,
        names=[
            'user_id','gender','nb_twitter_friends',
            'nb_twitter_followers'
        ]
    )


## Preprocessing

### Côté utilisateur

On commence par définir la maison des utilisateurs par le centroïde des lieux qu'ils ont visités.

In [24]:
def get_centroid(df):
    # Conversion en radians
    lat = np.radians(df['latitude'])
    lon = np.radians(df['longitude'])
    
    # Passage en cartésien
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    
    # Moyenne des coordonnées
    avg_x = x.mean()
    avg_y = y.mean()
    avg_z = z.mean()
    
    # Reconversion en Sphérique
    central_lon = np.arctan2(avg_y, avg_x)
    central_hyp = np.sqrt(avg_x**2 + avg_y**2)
    central_lat = np.arctan2(avg_z, central_hyp)
    
    return pd.Series({
        'lat_home': np.degrees(central_lat),
        'lon_home': np.degrees(central_lon)
    })

# Application par utilisateur
user_home = df_checkins.groupby('user_id').apply(get_centroid).reset_index()

On enrichit les checkins initiaux par les informations utiles sur les utilisateurs.

In [25]:
user_profiles = pd.merge(user_home, df_users[['user_id', 'nb_twitter_followers']], on='user_id')
df_final = pd.merge(df_checkins, user_profiles, on='user_id')

On calcule ensuite les distances parcourues pour chaque check-in.

In [26]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371.0  # Rayon de la Terre en km
    
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df_final['distance_km'] = haversine_vectorized(
    df_final['lat_home'], df_final['lon_home'],
    df_final['latitude'], df_final['longitude']
)

df_gravity = df_final[[
    'user_id', 'location_id', 'latitude', 'longitude', 
    'lat_home', 'lon_home', 'nb_twitter_followers', 'distance_km'
]]

In [28]:
df_gravity

,user_id,location_id,latitude,longitude,lat_home,lon_home,nb_twitter_followers,distance_km
0,886,4b5a91e5f964a520cbcb28e3,35.613889,139.724802,35.642628,139.743851,1140,3.629850
1,214,4c39423c1e06d13a1e6a793e,35.672565,139.484664,35.667098,139.591049,1533,9.629378
2,96,4b5dec1df964a520ed7329e3,35.609101,139.516304,35.629238,139.672151,1842,14.264055
3,1876,4b5da9e0f964a520bb6529e3,35.748805,139.719599,35.699900,139.742083,2361,5.804417
4,904,4b5a86faf964a520d3c928e3,35.677225,139.682471,35.675720,139.691770,759,0.856451
...,...,...,...,...,...,...,...,...
120154,1327,4b682bd6f964a520446a2be3,35.747081,139.623217,35.710814,139.678408,363,6.409505
120155,1048,4b5d21a0f964a520db5329e3,35.678520,139.744281,35.670277,139.727743,2752,1.752596
120156,1450,4b0a405ff964a520c22223e3,35.666567,139.726213,35.671289,139.717358,777,0.956914
120157,481,50c96352e4b0cc6a875c532a,35.701178,139.771038,35.664395,139.725070,215,5.828009


### Côté lieu

In [29]:
venue_stats = df_gravity.groupby('location_id').agg({
    'nb_twitter_followers': 'sum', # Somme de l'influence reçue
    'distance_km': 'mean',         # Distance moyenne parcourue par les visiteurs
    'user_id': 'count'             # Nombre total de check-ins (pour comparer)
}).rename(columns={'nb_twitter_followers': 'total_influence_score'})

In [ ]:
# --- Paramètres du modèle ---
# alpha = 1.0 (linéaire), alpha = 2.0 (loi de Newton classique)
# À Tokyo, la friction est faible grâce au métro, on peut tester 1.5
ALPHA = 1.5 
G = 1.0

# 1. Éviter la division par zéro (si distance_km est 0, on met un minimum de 100m)
df_gravity['distance_km'] = df_gravity['distance_km'].clip(lower=0.1)

# 2. Calculer la force d'attraction individuelle pour chaque check-in
# Force = (Followers) / (Distance^alpha)
df_gravity['attraction_force'] = G * (df_gravity['nb_twitter_followers'] / np.power(df_gravity['distance_km'], ALPHA))

# 3. Aggréger par lieu (Location)
venue_results = df_gravity.groupby('location_id').agg({
    'attraction_force': 'sum',          # Score de Newton (Popularité prédite)
    'nb_twitter_followers': 'sum',      # Masse totale reçue
    'user_id': 'count',                 # Nombre de check-ins réels
    'distance_km': 'mean',              # Distance moyenne parcourue
    'latitude': 'first',                # Pour cartographie
    'longitude': 'first'
}).rename(columns={
    'attraction_force': 'gravity_score',
    'user_id': 'visit_count',
    'nb_twitter_followers': 'total_followers_mass'
})

# 4. Normalisation pour faciliter la lecture (0 à 100)
venue_results['gravity_score_norm'] = (venue_results['gravity_score'] / venue_results['gravity_score'].max()) * 100

# 5. Trier par les lieux les plus "attractifs" selon Newton
top_venues = venue_results.sort_values(by='gravity_score', ascending=False)

print("--- Top 10 des lieux par score de gravité à Tokyo ---")
print(top_venues[['gravity_score_norm', 'visit_count', 'total_followers_mass']].head(10))

--- Top 10 des lieux par score de gravité à Tokyo ---
                          gravity_score_norm  visit_count  \
location_id                                                 
4b19f917f964a520abe623e3          100.000000         2720   
4b416f4cf964a520a6c625e3           64.973742           89   
4b0587a6f964a5203e9e22e3           64.752644          945   
4b19f837f964a520a2e623e3           54.564638          417   
4b0587a6f964a5203d9e22e3           52.289111         2471   
4b283d6ff964a520679124e3           51.540795           81   
4b46ad4cf964a520ad2626e3           41.221563          203   
4b56dcf4f964a5205f1d28e3           37.192852           91   
4b5edaf0f964a520149b29e3           36.613935          441   
4bf29ddea32e20a14266d557           34.217060          212   

                          total_followers_mass  
location_id                                     
4b19f917f964a520abe623e3               2962607  
4b416f4cf964a520a6c625e3                310950  
4b0587a6f964a5203

In [17]:
correlation = venue_results['gravity_score'].corr(venue_results['visit_count'])
print(f"\nCorrélation entre le modèle de Newton et les visites réelles : {correlation:.4f}")


Corrélation entre le modèle de Newton et les visites réelles : 0.6540


In [19]:
import statsmodels.api as sm
import numpy as np

# 1. Préparation des données pour le test
# On travaille sur les lieux (venues)
test_df = venue_results.copy()

# On évite les log(0)
test_df = test_df[test_df['total_followers_mass'] > 0]
test_df = test_df[test_df['distance_km'] > 0]

# 2. Transformation Logarithmique
# Variable à expliquer (Y) : Nombre de visites réelles
# Variables explicatives (X) : Masse des followers et Distance
test_df['log_visits'] = np.log(test_df['visit_count'])
test_df['log_mass'] = np.log(test_df['total_followers_mass'])
test_df['log_dist'] = np.log(test_df['distance_km'])

# 3. Définition du modèle OLS (Ordinary Least Squares)
# Y = Intercept + beta1 * log_mass + beta2 * log_dist
X = test_df[['log_mass', 'log_dist']]
X = sm.add_constant(X) # Ajoute l'intercept (G dans Newton)
Y = test_df['log_visits']

model = sm.OLS(Y, X).fit()

# 4. Affichage des résultats
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             log_visits   R-squared:                       0.473
Model:                            OLS   Adj. R-squared:                  0.473
Method:                 Least Squares   F-statistic:                     9773.
Date:                Wed, 22 Apr 2026   Prob (F-statistic):               0.00
Time:                        18:43:57   Log-Likelihood:                -24171.
No. Observations:               21774   AIC:                         4.835e+04
Df Residuals:                   21771   BIC:                         4.837e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.2802      0.023   -100.232      0.0